In [20]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from collections import deque
import pandas as pd
import matplotlib.pyplot as plt

class TradingEnvironment(gym.Env):
    """A custom trading environment following Gymnasium API"""
    
    metadata = {'render_modes': ['human'], 'render_fps': 4}
    
    def __init__(self, data, initial_balance=10000, max_position=100,
                 transaction_cost=0.001, risk_aversion=0.5, 
                 reward_type='risk_adjusted'):
        """
        Args:
            data: DataFrame with price data (must have 'close' column)
            initial_balance: Starting cash balance
            max_position: Maximum position size (absolute value)
            transaction_cost: Cost per trade as percentage
            risk_aversion: Lambda coefficient for risk penalty
            reward_type: Type of reward function ('dense', 'sparse', 'risk_adjusted')
        """
        super().__init__()
        
        # Store parameters
        self.data = data.reset_index(drop=True)
        self.initial_balance = initial_balance
        self.max_position = max_position
        self.transaction_cost = transaction_cost
        self.risk_aversion = risk_aversion
        self.reward_type = reward_type
        
        # Track episode state
        self.current_step = 0
        self.cash = initial_balance
        self.position = 0  # Positive = long, Negative = short
        self.portfolio_value = initial_balance
        self.peak_portfolio_value = initial_balance
        self.max_drawdown = 0
        
        # Track history for diagnostics
        self.value_history = [initial_balance]
        self.position_history = [0]
        self.reward_history = []
        self.drawdown_history = [0]
        self.returns_history = []
        
        # For volatility calculation
        self.return_window = deque(maxlen=20)  # Last 20 returns for volatility
        
        # Action space: -1 to 1 (sell to buy)
        # 0 = hold, positive = buy, negative = sell
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(1,), dtype=np.float32
        )
        
        # Observation space: [price, position, cash, portfolio_value, time]
        # Normalized values for better learning
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(5,), dtype=np.float32
        )
        
        # For rendering
        self.fig = None
        
    def reset(self, seed=None, options=None):
        """Reset the environment to initial state"""
        super().reset(seed=seed)
        
        self.current_step = 0
        self.cash = self.initial_balance
        self.position = 0
        self.portfolio_value = self.initial_balance
        self.peak_portfolio_value = self.initial_balance
        self.max_drawdown = 0
        
        # Reset history
        self.value_history = [self.initial_balance]
        self.position_history = [0]
        self.reward_history = []
        self.drawdown_history = [0]
        self.returns_history = []
        self.return_window.clear()
        
        observation = self._get_observation()
        info = self._get_info()
        
        return observation, info
    
    def step(self, action):
        """Execute one time step within the environment"""
        # Ensure action is within bounds
        action = np.clip(action, -1.0, 1.0)[0]
        
        # Store previous portfolio value for reward calculation
        prev_portfolio_value = self.portfolio_value
        
        # Execute trade based on action
        self._execute_trade(action)
        
        # Update portfolio value
        self._update_portfolio_value()
        
        # Calculate reward based on reward_type
        reward = self.calculate_reward(prev_portfolio_value)
        
        # Update step
        self.current_step += 1
        
        # Check if episode is done
        done = self.current_step >= len(self.data) - 1
        
        # Get observation and info
        observation = self._get_observation()
        info = self._get_info()
        info['reward'] = reward
        info['raw_pnl'] = self.portfolio_value - prev_portfolio_value
        
        # Store history
        self.value_history.append(self.portfolio_value)
        self.position_history.append(self.position)
        self.reward_history.append(reward)
        self.drawdown_history.append(self.max_drawdown)
        
        return observation, reward, done, False, info
    
    def _execute_trade(self, action):
        """
        Execute a trade based on the action signal.
        Action: -1 to 1 where:
            -1 = sell all (go to max short)
            0 = hold
            1 = buy all (go to max long)
        """
        current_price = self.data.iloc[self.current_step]['close']
        
        # Determine target position
        target_position = action * self.max_position
        
        # Calculate position change
        position_change = target_position - self.position
        
        if abs(position_change) > 0:
            # Calculate transaction cost
            trade_value = abs(position_change) * current_price
            transaction_cost = trade_value * self.transaction_cost
            
            # Update cash (pay transaction cost)
            self.cash -= transaction_cost
            
            # Update position
            self.position = target_position
    
    def _update_portfolio_value(self):
        """Update portfolio value based on current position and price"""
        current_price = self.data.iloc[self.current_step]['close']
        
        # Position value
        position_value = self.position * current_price
        
        # Total portfolio value
        self.portfolio_value = self.cash + position_value
    
    def _get_observation(self):
        """Get current observation"""
        if self.current_step >= len(self.data):
            return np.zeros(self.observation_space.shape, dtype=np.float32)
        
        price = self.data.iloc[self.current_step]['close']
        
        # Normalize observation components for better learning
        # Price: normalized by initial price
        norm_price = price / self.data.iloc[0]['close']
        
        # Position: normalized by max position
        norm_position = self.position / self.max_position if self.max_position > 0 else 0
        
        # Cash: normalized by initial balance
        norm_cash = self.cash / self.initial_balance
        
        # Portfolio value: normalized by initial balance
        norm_portfolio = self.portfolio_value / self.initial_balance
        
        # Time: normalized by total steps
        norm_time = self.current_step / len(self.data)
        
        observation = np.array([
            norm_price,
            norm_position,
            norm_cash,
            norm_portfolio,
            norm_time
        ], dtype=np.float32)
        
        return observation
    
    def _get_info(self):
        """Get additional information for diagnostics"""
        sharpe = self._calculate_sharpe_ratio()
        sortino = self._calculate_sortino_ratio()
        
        return {
            'step': self.current_step,
            'portfolio_value': self.portfolio_value,
            'cash': self.cash,
            'position': self.position,
            'max_drawdown': self.max_drawdown,
            'sharpe_ratio': sharpe,
            'sortino_ratio': sortino
        }
    
    def calculate_reward(self, prev_portfolio_value):
        """
        Calculate reward based on the configured reward type.
        
        This is the core method that needs to be engineered carefully.
        The agent will optimize EXACTLY what this function returns.
        """
        
        # Current portfolio value
        current_value = self.portfolio_value
        price = self.data.iloc[self.current_step]['close']
        
        # Calculate incremental PnL (dense reward)
        incremental_pnl = current_value - prev_portfolio_value
        
        # Calculate returns for volatility
        if prev_portfolio_value > 0:
            returns = (current_value - prev_portfolio_value) / prev_portfolio_value
            self.returns_history.append(returns)
            self.return_window.append(returns)
        
        if self.reward_type == 'sparse':
            # Reward only at end of episode
            if self.current_step >= len(self.data) - 1:
                # End of episode reward
                reward = (current_value - self.initial_balance) / self.initial_balance
            else:
                reward = 0.0
                
        elif self.reward_type == 'dense':
            # Simple dense reward based on PnL
            reward = incremental_pnl / self.initial_balance  # Normalized
            
        elif self.reward_type == 'risk_adjusted':
            # Advanced risk-adjusted reward
            reward = self._calculate_risk_adjusted_reward(
                incremental_pnl, prev_portfolio_value, current_value, price
            )
            
        else:
            # Default to risk-adjusted
            reward = self._calculate_risk_adjusted_reward(
                incremental_pnl, prev_portfolio_value, current_value, price
            )
        
        # Ensure reward is numerically stable
        reward = np.clip(reward, -10.0, 10.0).astype(np.float32)
        
        return reward
    
    def _calculate_risk_adjusted_reward(self, incremental_pnl, prev_value, current_value, price):
        """
        Calculate risk-adjusted reward using the formula:
        Reward = Normalized(PnL) - λ * Risk_penalty
        
        Where Risk_penalty can include multiple components.
        """
        
        # Normalize PnL component (scale to make learning stable)
        normalized_pnl = incremental_pnl / self.initial_balance
        
        # Initialize all risk components with default values
        risk_values = {
            'volatility': 0.0,
            'drawdown': 0.0,
            'inventory': 0.0,
            'overtrading': 0.0,
            'ruin_risk': 0.0
        }
        
        # 1. Volatility penalty (encourages smooth returns)
        if len(self.return_window) >= 5:  # Need enough data
            volatility = np.std(list(self.return_window))
            # Scale volatility to be comparable to normalized PnL
            risk_values['volatility'] = volatility * 100  # Convert to percentage scale
        
        # 2. Drawdown penalty (prevents large losses)
        current_drawdown = self._calculate_drawdown(current_value)
        # Penalize when drawdown exceeds 2%
        if current_drawdown > 0.02:
            risk_values['drawdown'] = current_drawdown * 200  # Amplify penalty for large drawdowns
        
        # 3. Inventory penalty (prevents holding large positions)
        # Position value relative to portfolio
        position_value = abs(self.position * price)
        if position_value > 0:
            position_ratio = position_value / current_value
            # Penalize positions > 50% of portfolio
            if position_ratio > 0.5:
                risk_values['inventory'] = (position_ratio - 0.5) * 100
        
        # 4. Transaction cost awareness (overtrading penalty)
        if len(self.position_history) > 10:
            # Calculate how often position changes sign (excessive trading)
            position_changes = np.diff(self.position_history[-10:])
            sign_changes = np.sum(np.abs(np.sign(position_changes)))
            if sign_changes > 5:  # More than 5 sign changes in last 10 steps
                risk_values['overtrading'] = 0.1 * sign_changes
        
        # 5. Risk of ruin penalty (prevent catastrophic losses)
        if current_value < self.initial_balance * 0.8:  # Lost more than 20%
            risk_values['ruin_risk'] = (self.initial_balance * 0.8 - current_value) / self.initial_balance * 100
        
        # Calculate total risk penalty (weighted sum)
        risk_weights = {
            'volatility': 0.3,
            'drawdown': 0.4,
            'inventory': 0.2,
            'overtrading': 0.05,
            'ruin_risk': 0.05
        }
        
        total_risk_penalty = 0
        for risk_name in risk_values:
            total_risk_penalty += risk_values[risk_name] * risk_weights.get(risk_name, 0.1)
        
        # Apply risk aversion coefficient (λ)
        risk_adjusted_reward = normalized_pnl - (self.risk_aversion * total_risk_penalty)
        
        # Store diagnostics
        self._store_diagnostics(normalized_pnl, risk_values, total_risk_penalty)
        
        return risk_adjusted_reward
    
    def _calculate_drawdown(self, current_value):
        """Calculate current drawdown from peak"""
        if current_value > self.peak_portfolio_value:
            self.peak_portfolio_value = current_value
        
        drawdown = (self.peak_portfolio_value - current_value) / self.peak_portfolio_value
        self.max_drawdown = max(self.max_drawdown, drawdown)
        
        return drawdown
    
    def _store_diagnostics(self, normalized_pnl, risk_values, total_risk_penalty):
        """Store reward components for analysis"""
        if not hasattr(self, 'reward_components'):
            self.reward_components = []
        
        components_dict = {'normalized_pnl': normalized_pnl}
        components_dict.update(risk_values)  # Add all risk values
        components_dict['total_risk_penalty'] = total_risk_penalty
        
        self.reward_components.append(components_dict)
    
    def _calculate_sharpe_ratio(self):
        """Calculate Sharpe ratio from returns history"""
        if len(self.returns_history) < 2:
            return 0
        
        returns = np.array(self.returns_history)
        mean_return = np.mean(returns)
        std_return = np.std(returns)
        
        if std_return == 0:
            return 0
        
        # Annualized Sharpe ratio (assuming 252 trading days)
        sharpe = (mean_return / std_return) * np.sqrt(252)
        return sharpe
    
    def _calculate_sortino_ratio(self):
        """Calculate Sortino ratio (downside risk only)"""
        if len(self.returns_history) < 2:
            return 0
        
        returns = np.array(self.returns_history)
        mean_return = np.mean(returns)
        
        # Only consider negative returns for downside deviation
        downside_returns = returns[returns < 0]
        if len(downside_returns) == 0:
            return 0
        
        downside_std = np.std(downside_returns)
        
        if downside_std == 0:
            return 0
        
        # Annualized Sortino ratio
        sortino = (mean_return / downside_std) * np.sqrt(252)
        return sortino
    
    def get_performance_metrics(self):
        """Get comprehensive performance metrics"""
        if len(self.value_history) < 2:
            return {}
        
        returns = np.diff(self.value_history) / self.value_history[:-1]
        
        # Calculate profit factor safely
        positive_returns = np.sum(returns[returns > 0])
        negative_returns = np.sum(returns[returns < 0])
        profit_factor = 0
        if negative_returns != 0:
            profit_factor = abs(positive_returns / negative_returns)
        
        metrics = {
            'total_return': (self.value_history[-1] - self.initial_balance) / self.initial_balance * 100,
            'max_drawdown': self.max_drawdown * 100,
            'sharpe_ratio': self._calculate_sharpe_ratio(),
            'sortino_ratio': self._calculate_sortino_ratio(),
            'volatility': np.std(returns) * np.sqrt(252) * 100 if len(returns) > 0 else 0,
            'win_rate': np.sum(returns > 0) / len(returns) * 100 if len(returns) > 0 else 0,
            'profit_factor': profit_factor,
            'total_trades': len(np.where(np.diff(self.position_history) != 0)[0]) if len(self.position_history) > 1 else 0,
            'final_portfolio_value': self.value_history[-1]
        }
        
        return metrics
    
    def close(self):
        """Close the environment"""
        if self.fig is not None:
            plt.close(self.fig)
            self.fig = None


def test_reward_functions():
    """Test different reward functions to see their effects"""
    import pandas as pd
    import numpy as np
    
    # Create synthetic price data
    np.random.seed(42)
    n_steps = 100
    prices = 100 + np.cumsum(np.random.randn(n_steps) * 2)
    data = pd.DataFrame({'close': prices})
    
    reward_types = ['sparse', 'dense', 'risk_adjusted']
    risk_aversions = [0.0, 0.5, 1.0]
    
    results = []
    
    for reward_type in reward_types:
        for risk_aversion in risk_aversions:
            print(f"\n{'='*60}")
            print(f"Testing: {reward_type} reward, λ={risk_aversion}")
            print('='*60)
            
            # Create environment
            env = TradingEnvironment(
                data=data,
                initial_balance=10000,
                max_position=10,
                transaction_cost=0.001,
                risk_aversion=risk_aversion,
                reward_type=reward_type
            )
            
            # Run a simple random policy for testing
            obs, info = env.reset()
            done = False
            total_reward = 0
            
            while not done:
                # Random action for testing
                action = env.action_space.sample()
                
                obs, reward, done, truncated, info = env.step(action)
                total_reward += reward
            
            # Get performance metrics
            metrics = env.get_performance_metrics()
            
            # Analyze reward components - SAFE VERSION
            if reward_type == 'risk_adjusted' and hasattr(env, 'reward_components') and env.reward_components:
                print("\nReward Components (avg):")
                
                # Get all possible keys from the first component
                all_keys = env.reward_components[0].keys()
                avg_rewards = {}
                
                for key in all_keys:
                    try:
                        values = [comp[key] for comp in env.reward_components if key in comp]
                        if values:  # Only calculate if we have values
                            avg_rewards[key] = np.mean(values)
                            print(f"  {key}: {avg_rewards[key]:.6f}")
                    except:
                        pass  # Skip if key doesn't exist
            
            print(f"\nPerformance Metrics:")
            print(f"  Total Reward: {total_reward:.4f}")
            print(f"  Total Return: {metrics['total_return']:.2f}%")
            print(f"  Max Drawdown: {metrics['max_drawdown']:.2f}%")
            print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.4f}")
            print(f"  Win Rate: {metrics['win_rate']:.1f}%")
            print(f"  Total Trades: {metrics['total_trades']}")
            
            results.append({
                'reward_type': reward_type,
                'risk_aversion': risk_aversion,
                'total_reward': total_reward,
                'total_return': metrics['total_return'],
                'max_drawdown': metrics['max_drawdown'],
                'sharpe_ratio': metrics['sharpe_ratio'],
                'win_rate': metrics['win_rate'],
                'total_trades': metrics['total_trades']
            })
            
            env.close()
    
    # Create summary DataFrame
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*100)
    print("SUMMARY OF REWARD FUNCTION PERFORMANCE")
    print("="*100)
    print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}' if abs(x) < 1000 else f'{x:.2f}'))
    
    # Print insights
    print("\n" + "="*100)
    print("KEY INSIGHTS:")
    print("="*100)
    print("1. Sparse rewards (end-of-day only) always give 0 cumulative reward during training")
    print("2. Dense rewards (PnL at each step) give highest total reward but often highest drawdown")
    print("3. Risk-adjusted rewards balance reward with risk (drawdown should be lower)")
    print("4. Higher λ (risk_aversion) should reduce drawdown but may also reduce returns")
    
    return results_df


# Quick test function to verify everything works
def quick_test():
    """Quick test to verify the environment works"""
    print("Running quick test...")
    
    # Create minimal data
    np.random.seed(42)
    data = pd.DataFrame({'close': 100 + np.cumsum(np.random.randn(50) * 2)})
    
    # Test each reward type quickly
    for reward_type in ['sparse', 'dense', 'risk_adjusted']:
        print(f"\nTesting {reward_type} reward...")
        
        env = TradingEnvironment(
            data=data,
            initial_balance=10000,
            max_position=10,
            transaction_cost=0.001,
            risk_aversion=0.5,
            reward_type=reward_type
        )
        
        obs, info = env.reset()
        print(f"  Initial observation (normalized): [{obs[0]:.3f}, {obs[1]:.3f}, {obs[2]:.3f}, {obs[3]:.3f}, {obs[4]:.3f}]")
        
        # Take a few steps
        for i in range(5):
            action = np.array([np.random.uniform(-1, 1)])  # Random action
            obs, reward, done, truncated, info = env.step(action)
            print(f"  Step {i+1}: reward={reward:.6f}, portfolio=${info['portfolio_value']:.2f}, position={info['position']:.1f}")
            
            if done:
                break
        
        env.close()
        print(f"  ✓ {reward_type} reward test passed")
    
    print("\n All quick tests passed!")


# Run the test
if __name__ == "__main__":
    # First run a quick test to make sure everything works
    quick_test()
    
    print("\n" + "="*100)
    print("NOW RUNNING COMPREHENSIVE REWARD FUNCTION TESTS")
    print("="*100)
    
    # Then run the comprehensive test
    test_results = test_reward_functions()

Running quick test...

Testing sparse reward...
  Initial observation (normalized): [1.000, 0.000, 1.000, 1.000, 0.000]
  Step 1: reward=0.000000, portfolio=$9273.99, position=-7.2
  Step 2: reward=0.000000, portfolio=$10606.67, position=6.0
  Step 3: reward=0.000000, portfolio=$9128.44, position=-8.5
  Step 4: reward=0.000000, portfolio=$11017.57, position=9.7
  Step 5: reward=0.000000, portfolio=$10563.57, position=5.4
  ✓ sparse reward test passed

Testing dense reward...
  Initial observation (normalized): [1.000, 0.000, 1.000, 1.000, 0.000]
  Step 1: reward=-0.060916, portfolio=$9390.84, position=-6.0
  Step 2: reward=-0.038788, portfolio=$9002.96, position=-9.9
  Step 3: reward=0.163801, portfolio=$10640.97, position=6.3
  Step 4: reward=-0.020921, portfolio=$10431.76, position=4.1
  Step 5: reward=0.004435, portfolio=$10476.11, position=4.6
  ✓ dense reward test passed

Testing risk_adjusted reward...
  Initial observation (normalized): [1.000, 0.000, 1.000, 1.000, 0.000]
  Step